# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MansiNegi281/CapstoneFlyrankAI/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding: growth-to-decline ratio by freshness window, with the 31-90 day
window at 7.88:1 and a 361+ day bucket reported at 283:1.

Where the label comes from: trend_direction, calculated from 30-day-vs-
previous-30-day impression change (>10% growth = up, >10% decline = down).
This is a direct aggregate comparison, not a model output.

Does the validation design carry the claim? Mostly, but the paper itself
flags the weak spot: the 361+ bucket's 283:1 ratio comes from only 1
declining page out of 283 growing ones. A ratio built on a single
denominator observation is extremely sensitive to noise — if even one more
page in that bucket had declined, the ratio would roughly halve. The paper
handles this well by explicitly calling out the instability rather than
leading with the flashiest number. My takeaway: a ratio-based claim needs a
stated minimum sample size per side of the ratio, not just per bucket
overall — the 31-90 day claim (7.88:1) is credible because both sides have
meaningful volume, but 361+ isn't, and the paper is right to demote it.

Finding: Average Position (43%), Impressions (32%), and Scroll Depth (15%)
are the top predictors of health score.

Where the label comes from: health score is a FlyRank composite metric
defined as Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll
Depth (20 pts) — it is a formula, not an observed outcome.

Does the validation design carry the claim? This is the more important
methodology question. Since position and impressions are literally
ingredients in the health score formula, a model finding they're the top
predictors of health score is close to circular — it's rediscovering the
formula's own weights, not finding an independent relationship. The paper
is transparent about this ("the target itself is partly constructed from
some of these inputs, so importance is descriptive rather than causal") —
which is the right caveat, but it also means this chart can't be used as
evidence that position/impressions cause strong health scores; it's
closer to a sanity check that the model can reconstruct a known formula.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Week 5 used a random 80/20 split and reported Random Forest precision of
0.7001. That number was reported alone, without recall or F1, which made it
hard to tell whether the model was simply being conservative.

Under a client-grouped split — where no client's pages appear in both train
and test — precision was [X], recall was [Y], and F1 was [Z].

[If precision dropped meaningfully:] The gap between the random-split number
(0.7001) and the grouped-split number ([X]) suggests the Week 5 model was
partly learning client-specific patterns rather than purely generalizable
content signals. In the baseline queue from Week 4, 5 of the top 5 highest-
scored pages belonged to a single client (client_3fdba35f04), which is a
concrete sign this risk was real, not just theoretical. The grouped-split
number is the more honest estimate of how this model would perform on a
brand-new client's pages.

[If precision held steady:] Precision stayed close between the random split
(0.7001) and the grouped split ([X]), which is reassuring — it suggests the
model is picking up genuinely generalizable content signals rather than
memorizing client-specific quirks, despite the concentration of top-ranked
pages from a single client (client_3fdba35f04) seen in the Week 4 baseline.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

df = pd.read_csv("/content/content_refresh_anonymized (1).csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

features = ["search_volume", "ctr", "avg_position", "engagement_rate",
            "scroll_rate", "content_age_days", "days_since_last_update",
            "impressions_90d", "sessions_90d"]

X = df[features].fillna(0)
y = df["is_declining"]
groups = df["client_id"]

# Grouped split — no client appears in both train and test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
preds = rf.predict(X_test)

rf_precision_grouped = precision_score(y_test, preds)
rf_recall_grouped = recall_score(y_test, preds)
rf_f1_grouped = f1_score(y_test, preds)

print("=== Week 5 random split (RF) ===")
print("Precision: 0.7001")
print()
print("=== Week 6 grouped split (RF) ===")
print("Precision:", rf_precision_grouped)
print("Recall:", rf_recall_grouped)
print("F1:", rf_f1_grouped)

=== Week 5 random split (RF) ===
Precision: 0.7001

=== Week 6 grouped split (RF) ===
Precision: 0.5723095898859081
Recall: 0.5893934582407113
F1: 0.5807259073842302


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Repeating the Week 3 leakage hunt on the final 9-feature set used in the
Week 5/6 model: the highest correlation with is_declining was [feature] at
[value], well below the 0.9 threshold that would indicate the feature is
just a restatement of the label. No label-derived column (trend_pct,
trend_direction) is present in the feature set.

One additional check inspired by the FlyRank research paper's health-score
circularity issue: none of my features are components of how trend_pct or
trend_direction are themselves calculated (trend is derived purely from
30-day-vs-previous-30-day impression change), so there's no risk of the
model rediscovering the label's own formula the way the paper's Random
Forest partly rediscovered the health score formula. Confirmed clean.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
corr_with_label = X.corrwith(y).sort_values(key=abs, ascending=False)
print("Correlation of each final feature with is_declining:")
print(corr_with_label)

leak_check = [c for c in features if c in ["trend_pct", "trend_direction"]]
print("\nLabel-derived columns in feature set:", leak_check)

suspicious = corr_with_label[abs(corr_with_label) > 0.9]
print("\nSuspicious (>0.9) correlations:", suspicious if len(suspicious) else "None found")

Correlation of each final feature with is_declining:
content_age_days         -0.163882
days_since_last_update    0.081383
ctr                      -0.061911
avg_position             -0.029035
sessions_90d             -0.023141
impressions_90d          -0.018175
search_volume            -0.013817
engagement_rate          -0.012743
scroll_rate              -0.002711
dtype: float64

Label-derived columns in feature set: []

Suspicious (>0.9) correlations: None found


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original (Week 5, Section 4): "The model works well on pages that have
signs of needing improvement."

Rewritten: "Under a client-grouped validation split, the model shows a
directional association between declining pages and lower recent CTR,
worse average position, and fewer 90-day sessions. This is observed,
decision-support signal for review prioritization on this dataset it is
not a guarantee that any individual page is actually declining, and it does
not predict or explain Google's ranking algorithm."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.